In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_lic = pd.read_parquet("data/data_licences/data_licences.parquet")
df_med = pd.read_csv("data/data_medailles/data_medailles.csv")


In [ ]:
######################## Fonctions pour créer les graphiques ############################


### Graphique nb de licenciés en fonction du temps et chaque courbe = un sport


def plot_licencies(df, annee, sport, nb_lic):
 
    table = df.pivot(index=annee, columns=sport, values=nb_lic).sort_index() # plus eff si index=année puis 1 col = 1 sport

    plt.figure(figsize=(9, 6))

    for sport in table.columns:
        plt.plot(table.index, table[sport], marker='o', label=sport)

    plt.xlabel("Année")
    plt.ylabel("Nombre de licenciés")
    plt.title("Évolution du nombre de licenciés par sport")
    plt.legend(title="Sport")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


### Graphique du taux d'accroissement du nb de licencés en fonction du temps et chaque courbe = un sport



def plot_taux_accroissement(df, annee, sport, nb_lic):
    
    table = df.pivot(index=annee, columns=sport, values=nb_lic).sort_index()

    taux_acc = table.pct_change() * 100

    plt.figure(figsize=(9, 6))
    for sport in taux_acc.columns:
        plt.plot(taux_acc.index, taux_acc[sport], marker='o', label=sport)

    plt.xlabel("Année")
    plt.ylabel("Taux d'accroissement (%)")
    plt.title("Taux d'accroissement du nombre de licenciés par sport")
    plt.legend(title="Sport")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


### Matrice de corrélation (mais de quoi par rapport à quoi?)


### Graphique avec en ordonnées à gauche le nb de licenciés, ordonées de droite le nb de méadailles et en abscisse le temps
# et un graphique correspond à un sport, donc je mets le code du sport en argument de ma fonction

import pandas as pd
import matplotlib.pyplot as plt

def plot_sport_licencies_medailles(df, sport_code, 
                                   annee, sport,
                                   lic, med):

    data = df[df[sport] == sport_code].sort_values(annee)

    if data.empty:
        print(f"Aucune donnée trouvée pour le sport '{sport_code}'.")
        return

    fig, ax1 = plt.subplots(figsize=(9, 6))

    # Axe de gauche = nb de licenciés
    color1 = 'tab:blue'
    ax1.set_xlabel("Année")
    ax1.set_ylabel("Nombre de licenciés", color=color1)
    ax1.plot(data[annee], data[lic], color=color1, marker='o', label='Licenciés')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, axis='x')

    # Axe de droite = nb médailles
    ax2 = ax1.twinx()
    color2 = 'tab:red'
    ax2.set_ylabel("Nombre de médailles", color=color2)
    ax2.plot(data[annee], data[med], color=color2, marker='s', linestyle='--', label='Médailles')
    ax2.tick_params(axis='y', labelcolor=color2)

    plt.title(f"Évolution du nombre de licenciés et de médailles - {sport_code}")
    fig.tight_layout()
    plt.show()



In [ ]:
df_lic.head()



In [ ]:
df_lic.columns

In [ ]:
df_med.columns

In [ ]:
df_med.head()

In [ ]:
df_clean_lic = df_lic[df_lic["Code_sport"] != "DIV"]

In [ ]:
# Appliquer les fonctions
def plot_licencies(df):


    # 1. Regroupement
    table = (
        df.groupby(["Année", "Code_sport"])["Licences annuelles"]
          .sum()
          .unstack()        # pivot
          .sort_index()
    )

    plt.figure(figsize=(25, 12))

    for sport in table.columns:
        plt.plot(table.index, table[sport], marker='o', label=sport)

    plt.xlabel("Année")
    plt.ylabel("Nombre de licences")
    plt.title("Évolution du nombre de licences par fédération")
    plt.legend(title="Code_sport")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_licencies(df_lic)

In [ ]:
def plot_taux_croissance(df):
    # 1. Tableau Année × Sport
    table = (
        df.groupby(["Année", "Code_sport"])["Licences annuelles"]
          .sum()
          .unstack()
          .sort_index()
    )

    # 2. Taux d’accroissement en %
    taux = table.pct_change() * 100

    # 3. Tracé
    plt.figure(figsize=(20, 10))

    for sport in taux.columns:
        plt.plot(taux.index, taux[sport], marker='o', label=sport)

    plt.axhline(0, color='black', linewidth=1)  # repère des 0%
    plt.xlabel("Année")
    plt.ylabel("Taux d'accroissement (%)")
    plt.title("Taux d'accroissement du nombre de licenciés par sport")
    plt.legend(title="Code_sport", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_taux_croissance(df_clean_lic)


In [ ]:
def plot_licences_medailles_jo_un_sport(df_lic, df_med, code_sport):
    """
    Pour un sport (code_sport) :
    - axe gauche  : nb de licenciés (courbe)
    - axe droit   : nb de médailles JO (barres pour 2020 et 2024)
    - abscisse    : année
    """
    # ----- Licences : agrégées par année -----
    lic = (
        df_lic[df_lic["Code_sport"] == code_sport]
        .groupby("Année")["Licences annuelles"]
        .sum()
        .rename("Licences")
    )

    # ----- Médailles JO 2020 / 2024 -----
    row = df_med[df_med["Code_sport"] == code_sport]
    if row.empty:
        raise ValueError(f"Aucune médaille JO pour le sport {code_sport}")
    row = row.iloc[0].fillna(0)

    med_2020 = row["2020_or"] + row["2020_argent"] + row["2020_bronze"]
    med_2024 = row["2024_or"] + row["2024_argent"] + row["2024_bronze"]

    med = pd.Series({2021: med_2020, 2024: med_2024}, name="Medailles")

    # ----- Fusion sur toutes les années concernées -----
    years = sorted(set(lic.index) | set(med.index))
    data = pd.DataFrame(index=years)
    data["Licences"] = lic.reindex(years)
    data["Medailles"] = med.reindex(years).fillna(0)

    # ----- Tracé -----
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()

    # Courbe des licenciés (axe gauche)
    ax1.plot(data.index, data["Licences"], marker="o", label="Licences")
    ax1.set_xlabel("Année")
    ax1.set_ylabel("Nombre de licenciés")
    ax1.tick_params(axis="y")

    # Barres des médailles (axe droit)
    ax2.bar(data.index, data["Medailles"], width=0.4, alpha=0.6, label="Médailles JO")
    ax2.set_ylabel("Nombre de médailles JO")
    ax2.tick_params(axis="y")

    # Titre et légende
    ax1.set_title(f"Licences et médailles JO pour le sport {code_sport}")
    l1, lab1 = ax1.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, lab1 + lab2, loc="upper left")

    ax1.grid(True)
    fig.tight_layout()
    plt.show()

plot_licences_medailles_jo_un_sport(df_clean_lic, df_med, 'NAT')